In [ ]:
import duckdb
import pandas as pd

# Step 1: Load CSV into a persistent file
with duckdb.connect(database='Module2projectdb.duckdb') as con:
    con.execute("CREATE TABLE customers_dataset AS SELECT * FROM read_csv_auto('/home/moswal/m2/M2Project/olist_customers_dataset.csv')")



                        customer_id                customer_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
2  4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   
3  b2b6027bc5c5109e529d4dc6358b12c3  259dac757896d24d7702b9acbbff3f3c   
4  4f2d8ab171c80ec8364f7c12e35b23ad  345ecd01c38d18a9036ed96c73b8d066   

  customer_zip_code_prefix          customer_city customer_state  
0                    14409                 franca             SP  
1                    09790  sao bernardo do campo             SP  
2                    01151              sao paulo             SP  
3                    08775        mogi das cruzes             SP  
4                    13056               campinas             SP  


In [ ]:
# Step 2: Connect to the persistent .duckdb file and fetch the data
con = duckdb.connect(database='Module2projectdb.duckdb')

# Fixed: Added 'users' table name to the SELECT query
df = con.execute("SELECT * FROM customers_dataset").df()

# View the first few rows of your Olist customers DataFrame
# print(df.head())

print("Number of rows:", len(df))

# identify unique Customers based upon unique Customer_IDs

# Get all unique customer IDs
unique_ids = df['customer_unique_id'].unique()

print("Number of rows of unique_customer_id:", len(unique_ids))
# Output will look like: ['06b8899923...' '18955e83d3...' '4e7b21e0d4...']


Number of rows: 99441
Number of rows of unique_customer_id: 96096


In [19]:
import streamlit as st
import duckdb
import pandas as pd

# Set up the dashboard page styling
st.set_page_config(page_title="Olist Customer Insights", page_icon="📊", layout="wide")

st.title("📊 Olist Customers Dashboard")
st.markdown("An interactive visualization of the Olist e-commerce customer dataset using DuckDB and Pandas.")

# --- STEP 1: Data Loading (Cached for performance) ---
@st.cache_data
def load_customer_data():
    # Connect to your persistent database file
    con = duckdb.connect(database='Module2projectdb.duckdb')
    
    # Fetch the required columns for analysis
    query = "SELECT customer_id, customer_unique_id, customer_state, customer_city FROM customers_dataset"
    df = con.execute(query).df()
    
    con.close()
    return df

# Display a clean loading message while data fetches
with st.spinner("Fetching data from Module2projectdb.duckdb..."):
    df = load_customer_data()


# --- STEP 2: KPI Metrics Section ---
st.subheader("Key Performance Indicators")
col1, col2, col3 = st.columns(3)

total_rows = len(df)
# Unique calculation using your variable logic
unique_customers = df['customer_unique_id'].nunique()
repeat_customers = total_rows - unique_customers

with col1:
    st.metric(label="Total Orders/Rows", value=f"{total_rows:,}")
with col2:
    st.metric(label="Unique Customers", value=f"{unique_customers:,}")
with col3:
    st.metric(label="Repeat Orders/Customers", value=f"{repeat_customers:,}", delta="- Accounts for loyalty")


# --- STEP 3: Visualizations ---
st.write("---")
st.subheader("Geographic Distribution of Unique Customers")

# Aggregate unique customers by state for plotting
state_data = df.groupby('customer_state')['customer_unique_id'].nunique().reset_index()
state_data.columns = ['State', 'Unique Customers Count']
state_data = state_data.sort_values(by='Unique Customers Count', ascending=False)

# Split screen into interactive Dataframe view and a visual chart
view_col1, view_col2 = st.columns([1, 2])

with view_col1:
    st.write("### Raw State Breakdown")
    st.dataframe(state_data, use_container_width=True, hide_index=True)

with view_col2:
    st.write("### Visual Chart (Top States)")
    # Render an interactive native bar chart
    st.bar_chart(
        data=state_data, 
        x='State', 
        y='Unique Customers Count', 
        color="#29b5e8", 
        use_container_width=True
    )


# --- STEP 4: Data Explorer ---
st.write("---")
with st.expander("🔍 View Raw Sample Dataset"):
    st.write("Showing the first 100 entries:")
    st.dataframe(df.head(100), use_container_width=True)

2026-06-02 20:44:50.376 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-02 20:44:50.378 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-02 20:44:50.379 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-02 20:44:50.381 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-02 20:44:50.383 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-02 20:44:50.384 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-02 20:44:50.385 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-02 20:44:50.386 No runtime found, using MemoryCacheStorageManager
2026-06-02 20:44:50.389 Thread 'MainThread':